# M60 scaling runs — Google Colab T4
60,439,040 parameters. Run qualification, then execute low/medium/high as independent seeded runs. Everything runs under local `/content`; no Google Drive connection is used.

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, subprocess, sys, tarfile, zipfile

UPLOAD_DIR = Path("/content")
WORKSPACE = Path("/content/toolcall_scaling_workspace")

# Ensure Colab saves the upload in a predictable location.
os.chdir(UPLOAD_DIR)

# Remove only older copies of this small project archive.
for old_archive in UPLOAD_DIR.glob("ToolCall-Scaling-Runs-Colab*.zip"):
    old_archive.unlink()

print("Upload ToolCall-Scaling-Runs-Colab.zip")
uploaded = files.upload()
uploaded_name = next(
    name for name in uploaded
    if name.startswith("ToolCall-Scaling-Runs-Colab") and name.endswith(".zip")
)
archive = UPLOAD_DIR / uploaded_name
assert archive.is_file(), archive

# Start with a clean, exact workspace.
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKSPACE)

# Normal expected archive layout.
PROJECT = WORKSPACE / "scaling_runs"

# Also tolerate an additional enclosing folder.
if not (PROJECT / "scripts/train.py").is_file():
    candidates = [
        notebook.parent.parent
        for notebook in WORKSPACE.rglob("m13_colab.ipynb")
        if (notebook.parent.parent / "scripts/train.py").is_file()
        and (notebook.parent.parent / "m30/m30_colab.ipynb").is_file()
        and (notebook.parent.parent / "m60/m60_colab.ipynb").is_file()
    ]
    assert candidates, "Could not locate scaling_runs inside the uploaded ZIP"
    PROJECT = candidates[0].resolve()

print("Archive:", archive)
print("Project:", PROJECT)
%cd {PROJECT}

## Local data, tokenizer, checkpoints, and resume archives
Before running the next cell, use Colab's Files pane to place `scaling_470m.tar.gz` (or `.zip`) in `/content`. The notebook installs it at `scaling_runs/data/scaling_470m`. The tokenizer belongs at `data/scaling_470m/tokenizer/toolcall_spm_32k.model`; a separately uploaded `/content/tokenizer.model` or `/content/toolcall_spm_32k.model` is copied there if the bundle does not already contain it. To resume, also upload any earlier `m60_*_seed42.zip` run archives to `/content`, then rerun this cell.

In [ ]:
DATA_LOCAL = PROJECT / 'data/scaling_470m'
RUNS_DIR = PROJECT / 'runs'

def _inside(root, member):
    target = (root / member).resolve()
    return target == root.resolve() or root.resolve() in target.parents

def extract_data_archive(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as handle:
            assert all(_inside(destination, item.filename) for item in handle.infolist()), 'Unsafe ZIP member'
            handle.extractall(destination)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as handle:
            assert all(_inside(destination, item.name) for item in handle.getmembers()), 'Unsafe tar member'
            handle.extractall(destination, filter='data')
    else:
        raise RuntimeError(f'Unsupported data archive: {archive_path}')

if not (DATA_LOCAL / 'COMPLETE').is_file():
    unpacked = Path('/content/scaling_470m')
    if (unpacked / 'COMPLETE').is_file():
        DATA_LOCAL.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(unpacked), str(DATA_LOCAL))
    else:
        candidates = [Path('/content') / name for name in ('scaling_470m.tar.gz', 'scaling_470m.tgz', 'scaling_470m.zip')]
        data_archive = next((path for path in candidates if path.is_file()), None)
        assert data_archive is not None, 'Upload scaling_470m.tar.gz or scaling_470m.zip into /content using the Files pane'
        staging = Path('/content/toolcall_scaling_data_unpack')
        if staging.exists(): shutil.rmtree(staging)
        extract_data_archive(data_archive, staging)
        matches = [marker.parent for marker in staging.rglob('COMPLETE') if marker.parent.name == 'scaling_470m']
        assert len(matches) == 1, f'Expected one scaling_470m bundle, found: {matches}'
        DATA_LOCAL.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(matches[0]), str(DATA_LOCAL))

TOKENIZER = DATA_LOCAL / 'tokenizer/toolcall_spm_32k.model'
if not TOKENIZER.is_file():
    tokenizer_uploads = [Path('/content/toolcall_spm_32k.model'), Path('/content/tokenizer.model')]
    tokenizer_source = next((path for path in tokenizer_uploads if path.is_file()), None)
    assert tokenizer_source is not None, f'Missing tokenizer: {TOKENIZER}'
    TOKENIZER.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(tokenizer_source, TOKENIZER)

RUNS_DIR.mkdir(parents=True, exist_ok=True)
for run_archive in sorted(Path('/content').glob('m60_*_seed42.zip')):
    with zipfile.ZipFile(run_archive) as handle:
        assert all(item.filename.startswith('runs/') and _inside(PROJECT, item.filename) for item in handle.infolist()), f'Unexpected run archive layout: {run_archive}'
        handle.extractall(PROJECT)
    print('Restored:', run_archive.name)

def export_run(run_name):
    output = Path('/content') / f'{run_name}.zip'
    if output.exists(): output.unlink()
    subprocess.run(['python', 'scripts/archive_run.py', run_name, '--runs-dir', str(RUNS_DIR), '--output', str(output), '--include-checkpoints'], check=True)
    print('Download and keep this archive for resume:', output)
    files.download(str(output))

print('Local data:', DATA_LOCAL)
print('Tokenizer:', TOKENIZER)
print('Local runs:', RUNS_DIR)

## Checkpoint 1 — environment, matrix, data, and model
Stop if the GPU is not a T4-class CUDA device, any exact parameter count is wrong, or the data verifier fails.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('CUDA:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['python', 'scripts/check_experiment_matrix.py'], check=True)
subprocess.run(['python', 'scripts/verify_data_bundle.py', 'data/scaling_470m'], check=True)
subprocess.run(['python', 'scripts/check_model.py', '--config', 'm60/configs/low_25m.json', '--forward-check'], check=True)
subprocess.run(['python', 'scripts/inspect_data.py', '--config', 'm60/configs/low_25m.json', '--tokenizer', str(TOKENIZER), '--samples', '1', '--sample-tokens', '128'], check=True)

## Checkpoint 2 — 1M-token qualification
Random data has no scientific meaning. Continue only if FP16 forward/backward, checkpointing, metrics, and final validation complete without OOM.

In [ ]:
subprocess.run(['python', 'scripts/create_synthetic_data.py'], check=True)
subprocess.run(['python', 'scripts/train.py', '--config', 'm60/configs/debug_1m.json', '--runs-dir', '/content/qualification_runs', '--device', 'cuda', '--resume', 'auto'], check=True)
subprocess.run(['python', 'scripts/summarize_run.py', 'm60_debug_1m_seed42', '--runs-dir', '/content/qualification_runs'], check=True)

## Live dashboard
TensorBoard updates while later training cells run. It reads the local runtime runs directory. Download each run archive after training; `/content` disappears when the runtime is deleted.

In [ ]:
%load_ext tensorboard
get_ipython().run_line_magic('tensorboard', f'--logdir {RUNS_DIR}')

## Preflight all three scientific points
This allocates each model and proves the shared 470M corpus is large enough. No training starts.

In [ ]:
CONFIGS = ["m60/configs/low_25m.json","m60/configs/medium_50m.json","m60/configs/high_100m.json"]
for config in CONFIGS:
    print('\nPREFLIGHT', config)
    subprocess.run(['python', 'scripts/train.py', '--config', config, '--runs-dir', str(RUNS_DIR), '--device', 'cuda', '--resume', 'auto', '--preflight-only'], check=True)

## LOW scientific point
This is an independent run. `--resume auto` resumes only this exact run name if it was interrupted.

In [ ]:
RUN_NAME = 'm60_low_25m_seed42'
try:
    subprocess.run(['python', 'scripts/train.py', '--config', 'm60/configs/low_25m.json', '--runs-dir', str(RUNS_DIR), '--device', 'cuda', '--resume', 'auto'], check=True)
    subprocess.run(['python', 'scripts/summarize_run.py', RUN_NAME, '--runs-dir', str(RUNS_DIR)], check=True)
finally:
    if (RUNS_DIR / RUN_NAME).is_dir(): export_run(RUN_NAME)

## MEDIUM scientific point
This is an independent run. `--resume auto` resumes only this exact run name if it was interrupted.

In [ ]:
RUN_NAME = 'm60_medium_50m_seed42'
try:
    subprocess.run(['python', 'scripts/train.py', '--config', 'm60/configs/medium_50m.json', '--runs-dir', str(RUNS_DIR), '--device', 'cuda', '--resume', 'auto'], check=True)
    subprocess.run(['python', 'scripts/summarize_run.py', RUN_NAME, '--runs-dir', str(RUNS_DIR)], check=True)
finally:
    if (RUNS_DIR / RUN_NAME).is_dir(): export_run(RUN_NAME)

## HIGH scientific point
This is an independent run. `--resume auto` resumes only this exact run name if it was interrupted.

In [ ]:
RUN_NAME = 'm60_high_100m_seed42'
try:
    subprocess.run(['python', 'scripts/train.py', '--config', 'm60/configs/high_100m.json', '--runs-dir', str(RUNS_DIR), '--device', 'cuda', '--resume', 'auto'], check=True)
    subprocess.run(['python', 'scripts/summarize_run.py', RUN_NAME, '--runs-dir', str(RUNS_DIR)], check=True)
finally:
    if (RUNS_DIR / RUN_NAME).is_dir(): export_run(RUN_NAME)

## Family completion check
All three summaries must say `completed` and each run must contain `checkpoint_final.pt`.

In [ ]:
import json
for run_name in ["m60_low_25m_seed42","m60_medium_50m_seed42","m60_high_100m_seed42"]:
    run = RUNS_DIR / run_name
    summary = json.loads((run / 'summary.json').read_text())
    final_checkpoint = run / 'checkpoints/checkpoint_final.pt'
    print(run_name, summary['status'], summary.get('best_validation_loss'), final_checkpoint.exists())
    assert summary['status'] == 'completed' and final_checkpoint.exists()